# Session Manager
> Manages one session: lazy flush, tree navigation, context building, fork/branch

In [ ]:
#| default_exp session_manager

In [ ]:
#| export
from typing import List, Optional, Any
from memexcode.core import gen_id, MessageEntry
from memexcode.session import SessionEntry
from memexcode.pluginspec import SessionStore, SessionInfo

In [ ]:
#| export
class SessionManager:
    """Manages one session: lazy flush, tree navigation, context building, fork/branch."""
    
    def __init__(self, store: SessionStore, session_id: str, cwd: str = ""):
        self._store = store  # SessionStore (has collection methods)
        self._session = store.get_session(session_id)  # SessionABC (per-session)
        self._session_id = session_id
        self.cwd = cwd
        self._leaf_id: Optional[str] = None
        self._pending: List[SessionEntry] = []
        self._flushed = False

    async def append(self, entry: SessionEntry) -> None:
        """Append entry with lazy flush - buffers until first assistant message."""
        if not entry.id: entry._entry.id = gen_id()
        if self._leaf_id: entry._entry.parentId = self._leaf_id
        self._pending.append(entry)
        self._leaf_id = entry.id
        
        if not self._flushed and getattr(entry, "role", None) == "assistant":
            await self._flush()
            self._flushed = True
        elif self._flushed: await self._session.append(entry)

    async def _flush(self) -> None:
        """Flush pending entries to store."""
        for entry in self._pending: await self._session.append(entry)
        self._pending.clear()

    async def navigate_to(self, entry_id: str) -> None:
        """Navigate to a specific entry in the tree (for branching)."""
        self._leaf_id = entry_id

    async def build_context(self) -> List[dict]:
        """Build LLM context from leaf back to root, handling compactions."""
        if not self._leaf_id: return []
        path = await self._session.get_path_to_root(self._leaf_id)
        result: List[dict] = []
        for entry in path:
            if entry.type == "compaction": result.append({"role": "system", "content": entry.summary})
            elif entry.type == "message": result.append(entry.to_dict())
        return result
    async def flush(self) -> None:
        """Force flush pending entries immediately."""
        await self._flush()
        self._flushed = True

    async def restore(self) -> bool:
        """Restore session from store. Finds latest entry, sets as leaf. Returns True if found."""
        entries = await self._session.load_all()
        if not entries: return False
        
        self._leaf_id = entries[-1].id  # last entry is latest
        self._pending.clear()  # fresh start, no pending
        self._flushed = True   # already persisted
        return True

    async def compact(self, first_kept_id: str, summary: str) -> None:
        """Compact history before first_kept_id."""
        await self._session.compact(first_kept_id, summary)

    async def fork(self, name: Optional[str] = None) -> SessionInfo:
        """Fork current session at leaf_id."""
        return await self._store.fork(self._session_id, self._leaf_id, name)

In [ ]:
!nbdev_export --path ../nbs/02_pluginspec.ipynb